# 04-02 ChatGLM3 推論實戰（2026 現代化版）

## 學習目標

1. 以 2026 統一慣例載入大型語言模型（`device_map='auto'`、`torch_dtype=torch.bfloat16`、`use_safetensors=True`）。
2. 用 `tokenizer.apply_chat_template()` 取代模型專屬的 `model.chat()` / `build_chat_input()`，理解對話模板（chat template）的可攜性。
3. 理解 ChatGLM3 的特殊 token 結構（`[gMASK]sop<|user|>…<|assistant|>`）以及為什麼應交由模板處理而非手刻。
4. 了解 bf16 相對於 fp16/fp32 的優勢，以及 safetensors 相對 pickle 的安全性與載入速度優勢。

## 前置需求

```
transformers>=4.46
torch>=2.4
safetensors>=0.4
accalerate>=1.0
```

## 與相鄰 notebook 的銜接

- 上一篇：[`01-8bits_training/chatglm3_8bit_train.ipynb`](../01-8bits_training/chatglm3_8bit_train.ipynb)——展示 8-bit 量化訓練。
- 下一篇：[`03-lora_training/chatglm3_lora_train.ipynb`](../03-lora_training/chatglm3_lora_train.ipynb)——加入 LoRA 微調流程。

本 notebook 聚焦**推論**，不含訓練，是量化訓練系列的「驗收環節」。

In [ ]:
# Environment pin — install once, then restart kernel
# Uncomment to install:
# !pip install -q \
#   "transformers>=4.46" \
#   "torch>=2.4" \
#   "accelerate>=1.0" \
#   "safetensors>=0.4"

import transformers, torch, accelerate, safetensors
print(f"transformers : {transformers.__version__}")
print(f"torch        : {torch.__version__}")
print(f"accelerate   : {accelerate.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
    print(f"VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. 模型 ID 設定

2026 統一慣例使用 HuggingFace Hub model ID，搭配可選的本機快取路徑（透過環境變數），讓 notebook 在任何環境都能執行。

- `HF_HOME` / `TRANSFORMERS_CACHE` 環境變數可指向本機快取目錄，無需修改程式碼。
- Hub ID 同時作為可重現的版本引用依據。

In [ ]:
import os
from pathlib import Path

# Use HF Hub model ID; set HF_HOME env var to point at a local cache directory if needed.
# Example (Linux/macOS):  export HF_HOME=/data/hf_cache
# Example (Windows):      set HF_HOME=D:\hf_cache
MODEL_ID = os.environ.get("MODEL_ID", "THUDM/chatglm3-6b")
print(f"Model ID: {MODEL_ID}")

## 2. 載入 Tokenizer

`AutoTokenizer.from_pretrained` 會自動從 Hub 下載或從本機快取讀取 tokenizer 設定。

ChatGLM3 的 tokenizer 包含自訂程式碼（`trust_remote_code=True`），這是 ZhipuAI 的 tokenizer 用來處理特殊 token（`[gMASK]`、`sop`、`<|user|>` 等）的必要選項。

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,  # required: ChatGLM3 ships custom tokenizer code
)
print(f"Vocab size     : {tokenizer.vocab_size}")
print(f"Chat template  : {'present' if tokenizer.chat_template else 'NOT FOUND'}")
print(f"BOS token      : {tokenizer.bos_token!r} (id={tokenizer.bos_token_id})")
print(f"EOS token      : {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")

## 3. 載入模型

### bf16 vs fp16 vs fp32

| 精度 | 位元 | 動態範圍 | VRAM(6B 模型) | 建議場景 |
|------|------|----------|---------------|----------|
| fp32 | 32 | 大 | ~24 GB | 精確訓練（需要時） |
| fp16 | 16 | 小，易溢位 | ~12 GB | Turing 以前的 GPU |
| bf16 | 16 | 與 fp32 相同指數位元 | ~12 GB | Ampere/Hopper（A100/H100/RTX 3090+）|

**bf16 優於 fp16 的關鍵**：bf16 的指數位元與 fp32 相同（8 位元），因此不會像 fp16 那樣在訓練中出現梯度下溢（gradient underflow），也不需要 loss scaling。推論時兩者速度相近，但 bf16 在大模型上更穩定。

### `device_map='auto'` 的語意

- `'auto'`：accelerate 自動依 VRAM 分配 layer——優先 GPU，不夠時溢位到 CPU RAM，最後才到磁碟（disk offload）。
- 這取代了零散的 `.cuda()`、`device=0`、`low_cpu_mem_usage=True` 等寫法，是 2026 載入大模型的統一慣例。

### `use_safetensors=True` 的優勢

- safetensors 格式不執行任何 Python 程式碼（pickle 有任意程式碼執行風險）。
- 載入速度快 2–5 倍（mmap 直接映射，不需反序列化）。
- 從 transformers 4.41 起，當 safetensors 檔案存在時預設自動使用；此處明確指定以確保行為。

2026 統一載入寫法：

```python
model = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    device_map="auto",          # auto layer placement + CPU/disk offload
    torch_dtype=torch.bfloat16, # stable 16-bit on Ampere+
    use_safetensors=True,       # safe + fast loading
)
```

> **VRAM 參考（ChatGLM3-6B）**
> - bf16 完整載入：約 12 GB VRAM
> - 若只有 8 GB VRAM：搭配 BitsAndBytesConfig 4-bit 量化可壓到約 4-5 GB（見本模組 01/03 notebook）
> - CPU-only：`device_map='cpu'`，速度慢但可執行

In [ ]:
import torch
from transformers import AutoModel

# VRAM requirement: ~12 GB for bf16 full-precision load.
# For 8 GB VRAM, use 4-bit quantization (see 01-8bits_training or 03-lora_training notebooks).
model = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,       # required: ChatGLM3 ships custom model code
    device_map="auto",            # auto-dispatch layers across GPU/CPU/disk
    torch_dtype=torch.bfloat16,   # stable 16-bit; prefer bf16 over fp16 on Ampere+
    use_safetensors=True,         # safe deserialization + faster mmap loading
)
model.eval()

print(f"Model dtype    : {next(model.parameters()).dtype}")
print(f"Model device   : {next(model.parameters()).device}")
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    print(f"VRAM allocated : {allocated:.2f} GB")

## 4. 理解 ChatGLM3 的對話 Token 結構

ChatGLM3 使用以下格式編碼多輪對話：

```
[gMASK] sop <|user|> \n {user_turn} <|assistant|> \n {assistant_turn} <eos>
```

其中：
- `[gMASK]`（token id `64790`）：生成式遮罩標記，是 ChatGLM 系列預訓練目標的遺跡。
- `sop`（`64792`，start of passage）：序列起始標記。
- `<|user|>`（`64795`）、`<|assistant|>`（`64796`）：角色標記。
- `<eos>`：序列結束。

以下程式碼展示如何**觀察**這個結構，目的是理解模板背後的機制——但在實際使用時，我們不應手刻這些 token（見第 5 節）。

In [ ]:
# Inspect the special token structure — for understanding only; do not hardcode in production.

# 1. Show special token IDs
special_tokens = {
    "[gMASK]": tokenizer.get_command("[gMASK]"),
    "sop"    : tokenizer.get_command("sop"),
    "<|user|>": tokenizer.get_command("<|user|>"),
    "<|assistant|>": tokenizer.get_command("<|assistant|>"),
}
print("Special token IDs:")
for name, tid in special_tokens.items():
    print(f"  {name:20s} -> {tid}")

# 2. Show what build_chat_input produces (internal mechanism)
input_ids_obj = tokenizer.build_chat_input("考試的技巧有哪些？", history=[], role="user")
input_ids = input_ids_obj["input_ids"][0].tolist()
print(f"\nbuild_chat_input token ids : {input_ids}")
print(f"Decoded                    : {tokenizer.decode(input_ids)}")

## 5. 2026 統一對話介面：`apply_chat_template()`

### 為什麼使用 `apply_chat_template()` 而非模型私有 API

`model.chat()` 與 `build_chat_input()` 是 ChatGLM 的私有方法，Llama/Mistral/Qwen 都不存在同名方法，且推論與訓練若使用不同的 prompt 組裝邏輯，容易造成 train/inference mismatch。2025 年後發布的模型（如 ChatGLM4、Qwen3）全部採用 `chat_template`，是跨模型的統一標準。

### 2026 統一寫法

```python
messages = [{"role": "user", "content": "考試的技巧有哪些？"}]
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,  # appends <|assistant|>\n so model knows to generate
)
```

優點：
- **跨模型可攜**：同一段程式碼，只換 `MODEL_ID` 就能用於 Llama3、Qwen3、Gemma3 等任何有 `chat_template` 的模型。
- **訓練/推論一致**：SFTTrainer 的 `formatting_func` 也呼叫同一模板，消除 mismatch。
- **多模態橋樑**：2026 的多模態模型（影像/音訊 token）也透過同一套模板機制注入，是通往 05-Multimodal 模組的核心抽象。

In [ ]:
# Construct a chat prompt using the portable apply_chat_template interface.

messages = [
    {"role": "user", "content": "考試的技巧有哪些？"},
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,  # appends the assistant role prefix so the model generates a reply
)

print("=== Prompt sent to model ===")
print(repr(prompt))
print()
print("=== Human-readable ===")
print(prompt)

## 6. 執行推論

`apply_chat_template` 產生的 prompt 字串直接透過 tokenizer 編碼後送入 `model.generate()`。

這與訓練側的流程對稱：SFTTrainer 也使用同一模板產生訓練樣本，確保訓練與推論格式完全一致。

### 生成參數說明

| 參數 | 值 | 說明 |
|------|----|------|
| `max_new_tokens` | 512 | 最多生成的新 token 數（不含輸入長度） |
| `do_sample` | True | 啟用取樣（否則是貪心解碼） |
| `temperature` | 0.7 | 控制隨機性；越低越保守 |
| `top_p` | 0.9 | nucleus sampling；只取累積機率前 90% 的 token |
| `repetition_penalty` | 1.1 | 抑制重複輸出 |

In [ ]:
import torch

# Tokenize the prompt
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[1]
print(f"Prompt token length: {input_len}")

# Generate
with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )

# Decode only the newly generated tokens (exclude the input prompt)
generated_ids = output_ids[0][input_len:]
response = tokenizer.decode(generated_ids, skip_special_tokens=True)

print("=== Model Response ===")
print(response)

## 7. 多輪對話示範

多輪對話的關鍵：將所有歷史訊息（包含 assistant 回覆）一起傳入 `apply_chat_template`，讓模板負責組裝完整上下文。

In [ ]:
def chat_once(model, tokenizer, messages: list[dict]) -> str:
    """Run one inference step given the full conversation history.

    Args:
        model: A loaded AutoModel (or AutoModelForCausalLM) in eval mode.
        tokenizer: The matching tokenizer with chat_template set.
        messages: List of {role, content} dicts representing the full conversation.

    Returns:
        The assistant's reply string.
    """
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][input_len:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)


# --- Multi-turn conversation demo ---
conversation = [
    {"role": "user", "content": "考試的技巧有哪些？"},
]

print("[User]", conversation[-1]["content"])
reply_1 = chat_once(model, tokenizer, conversation)
print("[Assistant]", reply_1)

# Append assistant reply to history and continue
conversation.append({"role": "assistant", "content": reply_1})
conversation.append({"role": "user", "content": "可以具體舉例說明記憶技巧嗎？"})

print()
print("[User]", conversation[-1]["content"])
reply_2 = chat_once(model, tokenizer, conversation)
print("[Assistant]", reply_2)

## 8. 系統提示（System Prompt）的使用

`apply_chat_template` 支援 `system` 角色，讓我們可以在對話開頭設定模型的行為指引，而無需了解各模型如何在內部編碼 system prompt。

In [ ]:
system_conversation = [
    {
        "role": "system",
        "content": "You are an experienced academic advisor specializing in study strategies. Answer concisely in Traditional Chinese.",
    },
    {"role": "user", "content": "考試的技巧有哪些？"},
]

print("[System]", system_conversation[0]["content"])
print("[User]", system_conversation[1]["content"])
reply_with_system = chat_once(model, tokenizer, system_conversation)
print("[Assistant]", reply_with_system)

## 小結

本 notebook 示範了三個核心現代化改動：

1. **統一載入慣例**：`device_map='auto'` + `torch_dtype=torch.bfloat16` + `use_safetensors=True` 是 2026 載入大型語言模型的標準寫法，且以 HF Hub model ID 取代硬路徑，讓 notebook 在任何環境可執行。

2. **對話模板標準化**：`tokenizer.apply_chat_template()` 是跨模型的統一介面，一段程式碼可攜至任何支援 chat template 的模型（Llama3、Qwen3、Gemma3 等），同時確保訓練與推論的 prompt 格式完全一致。

3. **Token 結構可見性**：透過觀察 ChatGLM3 的特殊 token（`[gMASK]`、`sop`、`<|user|>`、`<|assistant|>`），理解模板背後的機制，並體認「理解機制但不應手刻」的原則。

## 練習

1. 換用 `THUDM/chatglm3-6b-32k`（長上下文版本），觀察 tokenizer 的 chat_template 是否相同。
2. 嘗試替換為 `Qwen/Qwen3-7B` 或 `meta-llama/Meta-Llama-3-8B-Instruct`，僅修改 `MODEL_ID`，驗證 `apply_chat_template` 介面的跨模型可攜性。
3. 在 `system` role 中改用不同的角色設定（例如「數學家」、「翻譯員」），觀察回覆風格的差異。
4. 若 VRAM 不足 12 GB，參考 [`../01-8bits_training/chatglm3_8bit_train.ipynb`](../01-8bits_training/chatglm3_8bit_train.ipynb) 加入 `BitsAndBytesConfig` 進行 4-bit 量化推論。